In [2]:
# Patch pour le bug transformers 5.2.0 : video_processor_class_from_name ne gère pas les None
import importlib
import transformers.models.auto.video_processing_auto as _vpa
from transformers.models.auto.configuration_auto import model_type_to_module_name

def _patched_video_processor_class_from_name(class_name: str):
    for module_name, extractors in _vpa.VIDEO_PROCESSOR_MAPPING_NAMES.items():
        if extractors is None:
            continue
        if class_name in extractors:
            module_name = model_type_to_module_name(module_name)
            module = importlib.import_module(f".{module_name}", "transformers.models")
            if hasattr(module, class_name):
                return getattr(module, class_name)
    return None

_vpa.video_processor_class_from_name = _patched_video_processor_class_from_name
print("Patch appliqué avec succès.")


Patch appliqué avec succès.


In [ ]:
# Load model directly
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Utilisation du périphérique : {device}")

processor = AutoProcessor.from_pretrained("Qwen/Qwen3.5-4B")
model = AutoModelForImageTextToText.from_pretrained(
    "Qwen/Qwen3.5-4B",
    dtype=torch.bfloat16 if device == "cuda" else torch.float32,
    device_map="auto",
)


c:\Users\darkf\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Utilisation du périphérique : cuda


c:\Users\darkf\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\darkf\.cache\huggingface\hub\models--Qwen--Qwen3.5-4B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Fetching 2 files: 100%|██████████| 2/2 [05:16<00:00, 158.34s/it]
The fast path is not available be

In [ ]:
import threading
from PIL import Image
from transformers import TextIteratorStreamer

image = Image.open("./rue.png").convert("RGB")

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "url": image},
            {"type": "text", "text": "que vois-tu sur cette image ?"}
        ]
    },
]

inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

streamer = TextIteratorStreamer(processor.tokenizer, skip_prompt=True, skip_special_tokens=True)

generation_kwargs = dict(**inputs, max_new_tokens=200, streamer=streamer)
thread = threading.Thread(target=model.generate, kwargs=generation_kwargs)
thread.start()

for new_text in streamer:
    print(new_text, end="", flush=True)


thread.join()

Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


The Duser

    00umble  " " 25    000000.
  "   ?" "鹊  11这就 (一去 against

  (